# 12 Segmentation Validation

Валидация локального segmentation checkpoint на Thebe test cubes.

Базовые классы и функции берутся из `src/seismic_fault_recognition`. Данные должны уже лежать локально; скачивания в ноутбуке нет.

## 1. Bootstrap

Добавляем `src` в `sys.path`, чтобы ноутбук запускался прямо из репозитория.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## 2. Конфигурация эксперимента

Загружаем recipe и YAML-конфиг. Все пути редактируются в `configs/experiments`, а не в коде ноутбука.

In [ ]:
RECIPE_NAME = "segmentation_validation"
from seismic_fault_recognition.notebook_utils import (
    dataloader_kwargs,
    ensure_dir,
    load_notebook_context,
    print_context_summary,
    print_path_report,
    seed_everything,
    tuple3,
)

ctx = load_notebook_context(RECIPE_NAME, start=repo_root)
DATA = ctx.data
TRAINING = ctx.training
CHECKPOINTS = ctx.checkpoints
OUTPUTS = ctx.outputs

seed_everything(
    int(ctx.reproducibility.get("seed", TRAINING.get("seed", 42))),
    deterministic=bool(ctx.reproducibility.get("deterministic", True)),
    benchmark=bool(ctx.reproducibility.get("benchmark", False)),
)
print_context_summary(ctx)

## ClearML logging

Создаем ClearML task из recipe/config. Если `clearml` не установлен или отключен в конфиге, объект будет работать как no-op и обучение продолжится локально.

In [ ]:
from seismic_fault_recognition.clearml import (
    clearml_metric_logger,
    init_clearml_from_context,
    report_checkpoint_artifact,
    report_optimizer_lr,
)

clearml_run = init_clearml_from_context(ctx)
print(clearml_run.summary())

## 3. Быстрая проверка путей

Эта ячейка ничего не создает и не скачивает; она только показывает, какие локальные файлы уже доступны.

In [ ]:
print_path_report(
    {
        "train_seis": DATA.get("train_seis", ""),
        "train_fault": DATA.get("train_fault", ""),
        "val_seis": DATA.get("val_seis", ""),
        "val_fault": DATA.get("val_fault", ""),
        "test_seis": DATA.get("test_seis", ""),
        "test_fault": DATA.get("test_fault", ""),
        "output_dir": CHECKPOINTS.get("output_dir", ""),
    },
    ctx.repo_root,
)

## 4. Импорты

In [ ]:
import torch
from torch.utils.data import DataLoader

from seismic_fault_recognition.data import SeisFaultDataset, require_paths
from seismic_fault_recognition.losses import build_loss
from seismic_fault_recognition.models.factory import build_model_by_name
from seismic_fault_recognition.training import load_checkpoint
from seismic_fault_recognition.trainers import validate_segmentation_only

## 5. Локальные test-файлы и checkpoint

In [ ]:
paths = require_paths(
    {"test_seis": DATA["test_seis"], "test_fault": DATA["test_fault"], "checkpoint": CHECKPOINTS["input"]},
    base_dir=ctx.repo_root,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patch_size = tuple3(TRAINING["patch_size"])
print(f"device={device}, patch_size={patch_size}")

## 6. Test DataLoader

In [ ]:
test_ds = SeisFaultDataset(paths["test_seis"], paths["test_fault"], target_shape=patch_size, random_crop=False, seed=int(TRAINING["seed"]))
test_loader = DataLoader(
    test_ds,
    batch_size=1,
    **dataloader_kwargs(int(ctx.reproducibility.get("seed", TRAINING["seed"])), int(ctx.reproducibility.get("num_workers", TRAINING["num_workers"])), shuffle=False),
)
print(f"test samples={len(test_ds)}")

## 7. Модель и checkpoint

In [ ]:
model = build_model_by_name(ctx.recipe.model_variant, img_size=tuple3(TRAINING["roi_size"]), in_channels=1, out_channels=1).to(device)
checkpoint_strict = bool(ctx.validation.get("checkpoint_strict", True))
load_checkpoint(paths["checkpoint"], model=model, map_location=device, strict=checkpoint_strict)
loss_fn = build_loss(ctx.recipe.loss_profile).to(device)

## 8. Метрики

In [ ]:
metrics = validate_segmentation_only(
    model,
    test_loader,
    loss_fn,
    device=device,
    threshold=float(TRAINING["threshold"]),
    thresholds=ctx.validation.get("thresholds"),
)
clearml_run.report_metrics(metrics, iteration=0, title="Metrics", series_prefix="Validation")
print(metrics)